In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import Point

import climattr as eea

plt.rcParams.update({'font.size': 14})

In [ ]:

gdf = gpd.read_file('../supplementary/shapefiles/municipios/municipios.shp')
gdf['city_name'] = gdf['SIGLA_UF'] + '_' + gdf['city_name']

# Create figure and custom GridSpec layout (map on the left)
fig, [ax1, ax2] = plt.subplots(1, 2, figsize=(12, 6))

south_american_countries = [
    "Argentina",
    "Bolivia",
    "Brazil",
    "Chile",
    "Colombia",
    "Ecuador",
    "Guyana",
    "Paraguay",
    "Peru",
    "Suriname",
    "Uruguay",
    "Venezuela",
    "French Guiana", 
    "France"
]

world = gpd.read_file('../dengue_attribution/utils/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp')
south_america = world[world['SOVEREIGNT'].isin(south_american_countries)]

region = gpd.read_file('../dengue_attribution/utils/BR_UF_2022/BR_UF_2022.shp')
region_agg = region.dissolve(by='NM_REGIAO')
region_agg['region'] = region_agg.index
region_agg['region'] = region_agg['region'].replace({'Centro-oeste\n': 'Centre-West', 'Nordeste\n': 'Northeast', 'Norte': 'North', 'Sudeste\n': 'Southeast', 'Sul\n': 'South'})

prior_immunity =  pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv', parse_dates=['date_first_symptoms'])
prior_immunity['region'] = prior_immunity['region'].replace({'Middle-West': 'Centre-West'})

prior_immunity = prior_immunity.groupby(['date_first_symptoms', 'region'])[['Pr_0priorinf', 'Pr_1priorinf']].mean().reset_index()
prior_immunity = prior_immunity[prior_immunity['date_first_symptoms'] == pd.to_datetime('2024-01-01')]

prior_immunity = region_agg.set_index('region').join(prior_immunity.set_index('region'))

for ax_map, variable, label, cmap in zip([ax1, ax2], ['Pr_0priorinf', 'Pr_1priorinf'], ['never infected', 'infected at least once'], ['Spectral_r', 'Spectral']):
    # Brazil map on the left (spans both rows in column 0)
    south_america.plot(ax=ax_map, color='lightgrey', edgecolor='black', lw=0.5)
    plot = prior_immunity.plot(
        ax=ax_map, 
        column=variable, 
        vmin=0,
        vmax=1,
        cmap=cmap,
        legend=True,
        legend_kwds={"label": f"Proportion of individuals {label}", "orientation": "horizontal", "pad": 0.05, 'shrink': 0.85}
    )
    region_agg.plot(ax=ax_map, color='none', edgecolor='black', lw=1)
    ax_map.set_xlim([-75, -30])
    ax_map.set_ylim([-35, 5])
    ax_map.set_xticks([])
    ax_map.set_yticks([])

    # Access colorbar axis (usually the last one)
    cbar_ax = plot.figure.axes[-1]

plt.tight_layout()
plt.savefig('img/figure_immunity_2.pdf', bbox_inches='tight', dpi=300)

In [ ]:
df = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv', parse_dates=['date_first_symptoms'])
df['region'] = df['region'].replace({'Middle-West': 'Centre-West'})

regions = df['region'].unique()

fig, axarr = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)

for idx, region in enumerate(regions):
    ax = axarr.flatten()[idx]

    df[df['region'] == region].groupby('date_first_symptoms')['Pr_0priorinf'].mean().plot(ax=ax, label='Pr_0priorinf')
    df[df['region'] == region].groupby('date_first_symptoms')['Pr_1priorinf'].mean().plot(ax=ax, label='Pr_1priorinf')

    ax.set_title(region)
    ax.set_xlabel('Year')


axarr[0, 0].set_ylabel('Proportion of total')
axarr[1, 0].set_ylabel('Proportion of total')

# ---- LEGEND ----
legend_elements = [
    Line2D([0], [0], color='C0', lw=2, label='Not infected (Pr_0priorinf)'),
    Line2D([0], [0], color='C1', lw=2, label='Infected at least once (Pr_1priorinf)'),
]
fig.legend(handles=legend_elements, frameon=False, ncol=1, bbox_to_anchor=(0.99, 0.5))

fig.delaxes(axarr[-1, -1])

plt.tight_layout()
plt.savefig('img/immunity_city_with_priorinf.pdf', dpi=300, bbox_inches='tight')